In [16]:
# Instalar librerías
import gdown
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
import networkx as nx
import numpy as np

# ID del dataset
file_id = '18b8NP_O2TBlQ8mZSI5EpvCeU9r-QLzk_'
url = f'https://drive.google.com/uc?id={file_id}'

# Descargar el archivo físicamente al entorno de Colab
# Esto crea un archivo llamado 'datos_udea.xlsx' en la carpeta temporal de la izquierda
output = 'datos_udea.xlsx'
gdown.download(url, output, quiet=False)

# Leer el archivo
try:
    df = pd.read_excel(output, engine='openpyxl')
    print("\n✅ ¡Archivo de la UdeA cargado con éxito!")
    display(df.head())
except Exception as e:
    print("\n❌ Error al leer el Excel:")
    print(e)

Downloading...
From: https://drive.google.com/uc?id=18b8NP_O2TBlQ8mZSI5EpvCeU9r-QLzk_
To: /content/datos_udea.xlsx
100%|██████████| 12.7M/12.7M [00:00<00:00, 152MB/s]



✅ ¡Archivo de la UdeA cargado con éxito!


,MOVULTHIS,PACSEX,EDAD,MEDPAC,ACTCOD,ACTNOM,MOVULTFGR,MOVPOSPOS,MOVULTFUE,MOVULTDOC,MOVULTLIN,ESPNOM,DOSIS,VIA,FRECUENCIA,DURACION,DOSIS_mg,Freq_std
0,826697,M,68,AÃ±os,C107719,Atorvastatina 40 mg Tableta,2024-01-01 00:00:15,"40 (mg) miligramos, Oral, Cada 24 horas, por 3...",ME,17543039,1,MEDICINA DE URGENCIAS,40 (mg) miligramos,Oral,Cada 24 horas,por 30 DÃ­as,40.00,1.0
1,795532,M,67,AÃ±os,H013701,Hioscina butilbromuro 20 mg/1 mL Solucion inye...,2024-01-01 00:00:52,"20 (mg) miligramos, IntraVenosa, Cada 8 horas,...",ME,17543041,1,RESIDENTE MEDICINA INTERNA,20 (mg) miligramos,IntraVenosa,Cada 8 horas,por 1 Dia,20.00,3.0
2,795532,M,67,AÃ±os,T020701,Tramadol 50 mg/1 mL SoluciÃ³n inyectable Ampolla,2024-01-01 00:01:32,"25 (mg) miligramos, IntraVenosa, Dosis Ãºnica,...",ME,17543041,3,RESIDENTE MEDICINA INTERNA,25 (mg) miligramos,IntraVenosa,Dosis Ãºnica,por Dosis Unica,25.00,1.0
3,630475,M,33,AÃ±os,C047011,Clonidina 150 mcg Tableta,2024-01-01 00:10:31,"1 tableta(s), Oral, Cada 6 horas, por 30 DÃ­as",ME,17543058,4,MEDICO GENERAL,1 tableta(s),Oral,Cada 6 horas,por 30 DÃ­as,0.15,4.0
4,630475,M,33,AÃ±os,P026011,Prazosina 1 mg Tableta,2024-01-01 00:11:17,"2 (mg) miligramos, Oral, Cada 8 horas, por 30 ...",ME,17543058,5,MEDICO GENERAL,2 (mg) miligramos,Oral,Cada 8 horas,por 30 DÃ­as,2.00,3.0


In [17]:
### Cargar dataset original
dataset = pd.read_excel('datos_udea.xlsx')
dataset['row_id'] = dataset.index  # asegurar identificador único

### Filtrar dosis fijas (con frecuencia estandarizada válida)
df_fixed = dataset[dataset['Freq_std'] >= 0].copy()

### Renombrar columnas al formato estándar para DDC
df_ddc = df_fixed.rename(columns={
    'ACTNOM': 'drug',
    'DOSIS_mg': 'dose',
    'Freq_std': 'frequency'
})[['row_id', 'drug', 'dose', 'frequency']]

In [18]:
df_ddc.head()

,row_id,drug,dose,frequency
0,0,Atorvastatina 40 mg Tableta,40.00,1.0
1,1,Hioscina butilbromuro 20 mg/1 mL Solucion inye...,20.00,3.0
2,2,Tramadol 50 mg/1 mL SoluciÃ³n inyectable Ampolla,25.00,1.0
3,3,Clonidina 150 mcg Tableta,0.15,4.0
4,4,Prazosina 1 mg Tableta,2.00,3.0


In [19]:
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

In [20]:
### DDC: definición del modelo
class ddc_outlier():
    def __init__(self, alpha=0.9, metric='cosine'):
        self.alpha = alpha
        self.metric = metric
        self.pr = {}
        self.frequency = pd.DataFrame([])
        self.sim_matrix = np.zeros((1, 1))

    def fit(self, X):
        medication = pd.DataFrame(X, columns=['dose', 'freq'])
        medication['reg'] = 1
        MostFreq = medication[['reg','dose','freq']].groupby(['dose','freq']).agg(['count'])
        grouped = pd.DataFrame(MostFreq['reg']['count'])
        self.frequency = pd.DataFrame(grouped.values, columns=['count'])
        self.frequency['dose'] = [i[0] for i in grouped.index]
        self.frequency['freq'] = [i[1] for i in grouped.index]

        X = self.frequency[['dose','freq']].values.astype(float)
        try:
            if self.metric == 'similarity':
                self.sim_matrix = cosine_similarity(X, X)
            else:
                dists = pairwise_distances(X, X, metric=self.metric)
                self.sim_matrix = 1 / (1 + dists)
            graph = nx.from_numpy_array(self.sim_matrix)
            self.pr = nx.pagerank(graph, alpha=0.9, max_iter=1000,
                                  personalization=dict(self.frequency['count']))
        except Exception as e:
            print("Error en fit():", e)
            self.pr = dict(enumerate(np.zeros((len(X), 1)).flatten()))

    def predict(self, X):
        medication = pd.DataFrame(X, columns=['dose', 'freq'])
        medication['pr'] = 0.0
        for idx in self.frequency.index:
            row = self.frequency.loc[idx]
            match = (medication['dose'] == row['dose']) & (medication['freq'] == row['freq'])
            medication.loc[match, 'pr'] = self.pr.get(idx, 0)
        pr_threshold = np.mean(list(self.pr.values()))
        y_pred = medication['pr'].values
        y_pred[y_pred < (pr_threshold * self.alpha)] = -1
        y_pred[y_pred >= (pr_threshold * self.alpha)] = 1
        return y_pred

### Ejecutar DDC por medicamento
outliers = []

for drug in df_ddc['drug'].unique():
    subset = df_ddc[df_ddc['drug'] == drug]
    X = subset[['dose', 'frequency']].values

    if len(subset) < 10:
        continue

    model = ddc_outlier(alpha=0.5, metric='euclidean')
    model.fit(X)
    y_pred = model.predict(X)

    tmp = subset[['row_id']].copy()
    tmp['ddc_outlier'] = y_pred
    outliers.append(tmp)

# Unir resultados
df_ddc_out = pd.concat(outliers)

# Guardar resultado
df_ddc_out.to_csv('resultados_ddc_outliers.csv', index=False)
print("✅ Archivo guardado como resultados_ddc_outliers.csv")

✅ Archivo guardado como resultados_ddc_outliers.csv


In [21]:
df_ddc_out.head()

,row_id,ddc_outlier
0,0,1.0
127,127,1.0
139,139,1.0
560,560,1.0
619,619,1.0


In [24]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_ddc_out)

https://docs.google.com/spreadsheets/d/17M_MX9e1635OHPWmtjwcbLNm5GwMzO-aPHPoLI69D4U/edit#gid=0


In [25]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt # Ensure plt is imported here for consistency with the function

# Combinar ambos datasets para obtener las columnas necesarias
# Esta línea asume que df_ddc y df_ddc_out están disponibles de celdas ejecutadas previamente.
df_merged = pd.merge(df_ddc, df_ddc_out, on="row_id", how="inner")

# Función de graficado
def plot_ddc_outliers_interactive(drug_name_to_plot):
    df_drug = df_merged[df_merged['drug'] == drug_name_to_plot].copy()

    if df_drug.empty:
        print(f"No se encontraron datos para el medicamento: {drug_name_to_plot}")
        return

    freq_table = df_drug.groupby(['dose', 'frequency', 'ddc_outlier']).size().reset_index(name='count')

    min_size = 30
    max_size = 600
    if freq_table['count'].max() > freq_table['count'].min():
        freq_table['size'] = (
            (freq_table['count'] - freq_table['count'].min()) /
            (freq_table['count'].max() - freq_table['count'].min())
        ) * (max_size - min_size) + min_size
    else:
        freq_table['size'] = min_size

    fig, ax = plt.subplots(figsize=(8, 6))
    for outlier_value, color, label in [(-1, 'red', 'Outlier'), (1, 'blue', 'Normal')]:
        sub = freq_table[freq_table['ddc_outlier'] == outlier_value]
        ax.scatter(
            sub['frequency'],
            sub['dose'],
            s=sub['size'],
            c=color,
            label=label,
            alpha=0.6,
            edgecolors='black',
            linewidth=0.5
        )

    ax.set_xlabel("Frecuencia (dosis/día)")
    ax.set_ylabel("Dosis (mg)")
    ax.set_title(f"Distribución de prescripción DDC: {drug_name_to_plot}")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()

# Obtener la lista única de nombres de medicamentos
unique_drugs = sorted(df_merged['drug'].unique())

# Crear el widget interactivo
drug_selector = widgets.Dropdown(
    options=unique_drugs,
    description='Seleccionar Medicamento:',
    disabled=False,
)

# Enlazar el widget con la función de graficado
output_widget = widgets.Output()

def on_drug_change(change):
    with output_widget:
        clear_output(wait=True)
        plot_ddc_outliers_interactive(change.new)

drug_selector.observe(on_drug_change, names='value')

display(drug_selector, output_widget)

# Trazar el gráfico inicial para el primer medicamento de la lista
with output_widget:
    if unique_drugs:
        plot_ddc_outliers_interactive(unique_drugs[0])

Dropdown(description='Seleccionar Medicamento:', options=('Acetaminofen 10 mg/mL (1 g/100 mL) SoluciÃ³n inyect…

Output()